In [2]:
# In[1]:
import numpy as np
import math
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- CONFIGURATION ---
# !!! REPLACE THIS WITH THE ACTUAL PATH TO YOUR .TXT FILE !!!
file_path = 'hindi_sentences.txt'
# ---------------------

# 1. Load data
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        all_sentences = [line.strip() for line in f if line.strip()]
    print(f"Successfully loaded {len(all_sentences)} sentences.")
except FileNotFoundError:
    print(f"ERROR: File not found at '{file_path}'.")
    print("Please update the 'file_path' variable in this cell.")
    all_sentences = [] # Set to empty to avoid crashing next steps

# 2. Split data (simulating Assignment 1)
# Using an 80/10/10 split for 20,000 sentences
train_sents = all_sentences[:150000]
val_sents = all_sentences[150000:155000]
test_sents = all_sentences[155000:163148]

print(f"Data split:")
print(f"Training set:   {len(train_sents)} sentences")
print(f"Validation set: {len(val_sents)} sentences")
print(f"Testing set:    {len(test_sents)} sentences")

# Helper function to get bigrams from a list of sentences
def get_all_bigrams(sentence_list):
    all_bigrams = set()
    for sentence in sentence_list:
        tokens = sentence.split() # Using simple whitespace split
        if len(tokens) > 1:
            all_bigrams.update(zip(tokens, tokens[1:]))
    return all_bigrams

Successfully loaded 163148 sentences.
Data split:
Training set:   150000 sentences
Validation set: 5000 sentences
Testing set:    8148 sentences


In [6]:
# In[2]:
print("--- Starting Question 1: PMI Scores ---")

# 1. Build language models (counts) from TRAINING data
unigram_counts = Counter()
bigram_counts = Counter()
total_words = 0
total_bigrams = 0

print("Building unigram and bigram models from training data...")
for sentence in train_sents:
    tokens = sentence.split() # Simple whitespace tokenizer
    if not tokens:
        continue
    
    total_words += len(tokens)
    unigram_counts.update(tokens)
    
    sentence_bigrams = list(zip(tokens, tokens[1:]))
    total_bigrams += len(sentence_bigrams)
    bigram_counts.update(sentence_bigrams)

print(f"Training data stats:")
print(f"  Total words (unigrams): {total_words}")
print(f"  Total bigrams: {total_bigrams}")
print(f"  Unique unigrams (vocab size): {len(unigram_counts)}")
print(f"  Unique bigrams: {len(bigram_counts)}")

# 2. Define PMI calculation function
def calculate_pmi(w1, w2, unigram_counts, bigram_counts, total_words, total_bigrams):
    """Calculates PMI for a given bigram (w1, w2)."""
    
    count_w1 = unigram_counts.get(w1, 0)
    count_w2 = unigram_counts.get(w2, 0)
    count_w1_w2 = bigram_counts.get((w1, w2), 0)
    
    if count_w1 == 0 or count_w2 == 0 or count_w1_w2 == 0:
        return -math.inf
    
    prob_w1 = count_w1 / total_words
    prob_w2 = count_w2 / total_words
    prob_w1_w2 = count_w1_w2 / total_bigrams
    
    pmi = math.log2(prob_w1_w2 / (prob_w1 * prob_w2))
    return pmi

# 3. Compute PMI for all bigrams in validation set
print("\nCalculating PMI for validation set bigrams...")
val_bigrams = get_all_bigrams(val_sents)
val_pmi_scores = {}
for w1, w2 in val_bigrams:
    val_pmi_scores[(w1, w2)] = calculate_pmi(w1, w2, unigram_counts, bigram_counts, total_words, total_bigrams)

# 4. Compute PMI for all bigrams in testing set
print("Calculating PMI for testing set bigrams...")
test_bigrams = get_all_bigrams(test_sents)
test_pmi_scores = {}
for w1, w2 in test_bigrams:
    test_pmi_scores[(w1, w2)] = calculate_pmi(w1, w2, unigram_counts, bigram_counts, total_words, total_bigrams)

print("\n--- PMI Calculation Complete ---")

# --- 5. SAVE PMI RESULTS TO FILE ---

# Sort by PMI score (highest first)
sorted_val_pmi = sorted(val_pmi_scores.items(), key=lambda item: item[1], reverse=True)
val_pmi_file = 'pmi_validation.txt'
print(f"Saving all {len(sorted_val_pmi)} validation PMI scores to {val_pmi_file}...")
with open(val_pmi_file, 'w', encoding='utf-8') as f:
    f.write("PMI_Score\tBigram\n")
    for (w1, w2), pmi in sorted_val_pmi:
        f.write(f"{pmi:.8f}\t('{w1}', '{w2}')\n")

# Sort by PMI score (highest first)
sorted_test_pmi = sorted(test_pmi_scores.items(), key=lambda item: item[1], reverse=True)
test_pmi_file = 'pmi_testing.txt'
print(f"Saving all {len(sorted_test_pmi)} testing PMI scores to {test_pmi_file}...")
with open(test_pmi_file, 'w', encoding='utf-8') as f:
    f.write("PMI_Score\tBigram\n")
    for (w1, w2), pmi in sorted_test_pmi:
        f.write(f"{pmi:.8f}\t('{w1}', '{w2}')\n")

print("PMI results saved.")

--- Starting Question 1: PMI Scores ---
Building unigram and bigram models from training data...
Training data stats:
  Total words (unigrams): 2642394
  Total bigrams: 2492394
  Unique unigrams (vocab size): 135754
  Unique bigrams: 886387

Calculating PMI for validation set bigrams...
Calculating PMI for testing set bigrams...

--- PMI Calculation Complete ---
Saving all 54812 validation PMI scores to pmi_validation.txt...
Saving all 83782 testing PMI scores to pmi_testing.txt...
PMI results saved.


In [4]:
# In[3]:
print("--- Starting Question 2: TF-IDF Vectorization ---")

# 1. Initialize the TF-IDF Vectorizer
# We will use the default whitespace tokenizer, which works well for Hindi
# (token_pattern=r'(?u)\b\w\w+\b')
vectorizer = TfidfVectorizer()

# 2. Fit and transform the TRAINING data
# This learns the vocabulary and IDF scores
print("Fitting TF-IDF on training data...")
tfidf_train = vectorizer.fit_transform(train_sents)

# 3. Transform the VALIDATION data
# This uses the vocabulary and IDF scores learned from the train data
print("Transforming validation data...")
tfidf_val = vectorizer.transform(val_sents)

# 4. Transform the TESTING data
# This also uses the vocabulary and IDF scores learned from the train data
print("Transforming testing data...")
tfidf_test = vectorizer.transform(test_sents)

print("\n--- TF-IDF Vectorization Complete ---")
print(f"Shape of TF-IDF matrix (train): {tfidf_train.shape}")
print(f"Shape of TF-IDF matrix (validation): {tfidf_val.shape}")
print(f"Shape of TF-IDF matrix (test): {tfidf_test.shape}")
print(f"Total features (vocabulary size): {tfidf_train.shape[1]}")

--- Starting Question 2: TF-IDF Vectorization ---
Fitting TF-IDF on training data...
Transforming validation data...
Transforming testing data...

--- TF-IDF Vectorization Complete ---
Shape of TF-IDF matrix (train): (150000, 18017)
Shape of TF-IDF matrix (validation): (5000, 18017)
Shape of TF-IDF matrix (test): (8148, 18017)
Total features (vocabulary size): 18017


In [5]:
# In[4]:
print("--- Starting Question 3: Nearest Neighbors ---")

# --- 1. Process Validation Set ---
print("\nFinding nearest neighbors in Validation Set...")

# Calculate cosine similarity matrix (sentence vs. all other sentences)
# Shape: (num_val_sents, num_val_sents)
cosine_sim_val = cosine_similarity(tfidf_val)

# Set the diagonal to -1 (or any value < 0)
# This prevents a sentence from being its own nearest neighbor
np.fill_diagonal(cosine_sim_val, -1)

# Find the index of the highest similarity for each sentence
# np.argmax(axis=1) finds the index of the max value in each row
nearest_indices_val = np.argmax(cosine_sim_val, axis=1)

print("\n--- Validation Set Nearest Neighbors (Examples) ---")
for i in range(5):
    original_sentence = val_sents[i]
    neighbor_index = nearest_indices_val[i]
    neighbor_sentence = val_sents[neighbor_index]
    similarity = cosine_sim_val[i, neighbor_index]
    
    print(f"\nSentence {i} (Original):")
    print(f"  '{original_sentence[:100]}...'")
    print(f"Nearest Neighbor (Index {neighbor_index}, Sim: {similarity:.4f}):")
    print(f"  '{neighbor_sentence[:100]}...'")


# --- 2. Process Testing Set ---
print("\n\nFinding nearest neighbors in Testing Set...")

# Calculate cosine similarity matrix
cosine_sim_test = cosine_similarity(tfidf_test)

# Set the diagonal to -1
np.fill_diagonal(cosine_sim_test, -1)

# Find the index of the highest similarity for each sentence
nearest_indices_test = np.argmax(cosine_sim_test, axis=1)

print("\n--- Testing Set Nearest Neighbors (Examples) ---")
for i in range(5):
    original_sentence = test_sents[i]
    neighbor_index = nearest_indices_test[i]
    neighbor_sentence = test_sents[neighbor_index]
    similarity = cosine_sim_test[i, neighbor_index]
    
    print(f"\nSentence {i} (Original):")
    print(f"  '{original_sentence[:100]}...'")
    print(f"Nearest Neighbor (Index {neighbor_index}, Sim: {similarity:.4f}):")
    print(f"  '{neighbor_sentence[:100]}...'")

print("\n\n--- Nearest Neighbor Search Complete ---")

--- Starting Question 3: Nearest Neighbors ---

Finding nearest neighbors in Validation Set...

--- Validation Set Nearest Neighbors (Examples) ---

Sentence 0 (Original):
  '1....'
Nearest Neighbor (Index 1, Sim: 0.0000):
  'इसमें एंटीइंफ्लेमेंटरी प्रोपर्टी होती है।...'

Sentence 1 (Original):
  'इसमें एंटीइंफ्लेमेंटरी प्रोपर्टी होती है।...'
Nearest Neighbor (Index 4150, Sim: 0.6556):
  'ट्विटर पर पेस ने जाहिर की निराशा...'

Sentence 2 (Original):
  'इससे आपको सूजन और किसी तरह के दर्द से आराम मिलता है।...'
Nearest Neighbor (Index 2939, Sim: 0.5939):
  'इससे बहुत आराम मिलता है....'

Sentence 3 (Original):
  'अर्थराइटिस के मरीज के लिए ये काफी फायदेमंद होता है।...'
Nearest Neighbor (Index 2844, Sim: 0.3662):
  'फॉग लाइट गाड़ी की हेडलाइट के साथ ही नीचे की ओर फिट की जाती है।...'

Sentence 4 (Original):
  'सवाल : 11 नंबर ई-6 के नाले पर सेंट जोसफ स्कूल के तीन ब्रिज बने हैं, जिससे नाले की सफाई नहीं हो रही।...'
Nearest Neighbor (Index 4689, Sim: 0.4314):
  'स्टॉक में आमतौर पर सफेद, ब्राउन के क